# Dead Time Analysis — Non-Paralyzable Detector
**Method:** Fit the shifted exponential $p(\Delta t) = \lambda\, e^{-\lambda(\Delta t - \tau)}$ for $\Delta t \geq \tau$ to 6 inter-arrival time distributions.  
$\tau$ is **shared** across all runs (detector property); $\lambda_i$ is the **true rate** for run $i$.  
Individual source activities follow as $r_i = \lambda_i - \lambda_{i-1}$.

## The key property of the non-paralyzable model

For a non paralyzable detector with true rate $\lambda$, the inter-arrival time PDF between recorded events is:

$p(t) = \lambda \mathrm{e}^{-\lambda(t - \tau)}, \,\, t \geq \tau$

The crucial point: the exponential slope gives you the true rate $\lambda$ directly -- not the observed (reduced) rate. The date time only shifts the distribution; it does not change the decay constant. This is the most powerful tool.


## Cell 1 — Imports

In [21]:
import ROOT
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

ROOT.gStyle.SetOptStat(111)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

from pathlib import Path
import sys
import ipynbname
project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

# ROOT.gROOT.SetBatch(False)
# ROOT.gStyle.SetOptStat(0)
# ROOT.gStyle.SetOptFit(111)   # show fit params on plot
# ROOT.gStyle.SetPadGridX(True)
# ROOT.gStyle.SetPadGridY(True)
print('ROOT version:', ROOT.gROOT.GetVersion())

from visualization  import hist_quickShow

import ipynbname
import sys
from pathlib import Path
project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

ROOT version: 6.36.06


## Cell 2 — Load data
**Adjust** `FILES` to match your actual filenames.  
If all runs are in **one CSV** with a `run` column, replace the loading block with:
```python
df_all = pd.read_csv('all_runs.csv')
datasets = [df_all[df_all['run']==i]['time_s'].values for i in range(1, 7)]
```

In [22]:
N_RUNS = 6

data_path = project_root/'data/260527'
FILES = [list((data_path).glob(f'time_{i}*.csv'))[0] for i in [29,30,31,32,34,35]]
FILES

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_29_siphraAonly_Timing_Cs-137_1source.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_30_siphraAonly_Timing_Cs-137_2sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_31_siphraAonly_Timing_Cs-137_3sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_32_siphraAonly_Timing_Cs-137_4sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_34_siphraAonly_Timing_Cs-137_5sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_35_siphraAonly_Timing_Cs-137_6sources.csv')]

In [23]:
datasets = []
for i, fname in enumerate(FILES):
    df = pd.read_csv(fname)
    dt = df['time_s'].values.astype(float)
    dt = dt[dt > 0]          # drop any zeroes / artefacts
    datasets.append(dt)

print(f"{'Run':>4}  {'N events':>10}  {'min Δt (s)':>14}  {'mean Δt (s)':>14}  {'1/mean (Hz)':>14}")
print('-' * 62)
for i, dt in enumerate(datasets):
    print(f"{i+1:>4}  {len(dt):>10,}  {dt.min():>14.4e}  {dt.mean():>14.4e}  {1/dt.mean():>14.2f}")

 Run    N events      min Δt (s)     mean Δt (s)     1/mean (Hz)
--------------------------------------------------------------
   1     500,000      1.2000e-05      4.7683e-04         2097.20
   2     500,000      1.2000e-05      3.7966e-04         2633.97
   3     500,000      1.2000e-05      3.0813e-04         3245.43
   4     500,000      1.1000e-05      2.4317e-04         4112.39
   5     500,000      1.2000e-05      1.7851e-04         5601.92
   6     500,000      1.1000e-05      1.7030e-04         5871.83


## Cell 3 — First look: choose histogram binning
Key choices:
- **Lower edge = 0** so the dead-time onset is visible.
- **Bin width** should be small enough to resolve the onset at $\tau$ but not so small that bins are empty in the tail.
- A rule of thumb: aim for ≥ 10 counts/bin in the bulk of the distribution.

In [24]:
# Use the run with the HIGHEST rate to set the time axis
# (it has the shortest mean Δt and will drive the τ estimate)
means = np.array([dt.mean() for dt in datasets])

# Upper edge: 6 × mean of the LOWEST-rate run (run 1)
T_MAX  = 8.0 * means[0]

# Bin width: mean of the HIGHEST-rate run / 40  (empirical, adjust if needed)
BIN_W  = means[-1] / 15.0
N_BINS = int(T_MAX / BIN_W)

print(f'T_MAX  = {T_MAX:.4e} s')
print(f'BIN_W  = {BIN_W:.4e} s')
print(f'N_BINS = {N_BINS}')

T_MAX  = 3.8146e-03 s
BIN_W  = 1.1354e-05 s
N_BINS = 335


## Cell 4 — Build ROOT histograms

In [25]:
# COLORS = [ROOT.kBlue+1, ROOT.kRed+1, ROOT.kGreen+2,
          # ROOT.kMagenta+1, ROOT.kCyan+2, ROOT.kOrange+1]
COLORS = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]


hists = []
for i, dt in enumerate(datasets):
    h = ROOT.TH1D(f'h{i+1}', f'Run {i+1};#Delta t (s);Counts / bin',
                  N_BINS, 0.0, T_MAX)
    # h.SetDirectory(0)      # keep alive outside any TFile
    h.SetLineColor(COLORS[i])
    h.SetLineWidth(1)
    for t in dt:
        if 0 < t <= T_MAX:
            h.Fill(t)
    hists.append(h)
    print(f'Run {i+1}: {h.GetEntries():.0f} entries filled')

Run 1: 499981 entries filled
Run 2: 499981 entries filled
Run 3: 499907 entries filled
Run 4: 499779 entries filled
Run 5: 499771 entries filled
Run 6: 499740 entries filled


Warning in <TROOT::Append>: Replacing existing TH1: h1 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h2 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h3 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h4 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h5 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h6 (Potential memory leak).


## Cell 5 — Visual inspection (log-y)
On a log-y scale a shifted exponential is a straight line starting at $\tau$.  
Look for the **onset** (where counts drop to zero at small $\Delta t$) — that is $\tau$.

In [26]:
if ROOT.gROOT.FindObject('c_inspect'):
    ROOT.gROOT.FindObject('c_inspect').Close()

c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1600, 1200)
c_inspect.Divide(3, 2)
for i, h in enumerate(hists):
    c_inspect.cd(i + 1)
    h.SetLineWidth(2)
    h.Draw('HIST')
    c_inspect.cd(i+1).SetLogy()
c_inspect.Draw()

In [27]:
hist_quickShow(hists[::-1])

(<cppyy.gbl.TCanvas object at 0x586996bac620>, None)

In [28]:
# if ROOT.gROOT.FindObject('c_inspect'):
#     ROOT.gROOT.FindObject('c_inspect').Close()

# c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1400, 900)
# c_inspect.Divide(3, 2)
# for i, h in enumerate(hists):
#     c_inspect.cd(i + 1)
#     ROOT.gPad.SetLogy(1)
#     h.Draw('HIST')
# c_inspect.Draw()

## Cell 6 — Initial τ estimate and preliminary λᵢ
The global minimum of all inter-arrival times is a **lower bound** on $\tau$ (the true value is slightly higher due to finite statistics). We use it as a starting point.

For each run we get a rough $\lambda_i$ from $1 / (\bar{\Delta t}_i - \hat{\tau})$.

In [29]:
T_SKIP = 190e-6   # 50 µs — salta la región de pileup (~20 µs) con margen

# Trabajar solo con eventos por encima del corte
datasets_clean = [dt[dt > T_SKIP] for dt in datasets]

ns    = np.array([len(dt)   for dt in datasets_clean])
mins  = np.array([dt.min()  for dt in datasets_clean])
means = np.array([dt.mean() for dt in datasets_clean])

# Estima τ para cada run: posición del bin máximo por encima de T_SKIP
# (para una exponencial desplazada, el modo = τ)
tau_estimates = []
for dt in datasets_clean:
    t_max_hist = np.percentile(dt, 99)
    counts, edges = np.histogram(dt, bins=300, range=(T_SKIP, t_max_hist))
    centers = (edges[:-1] + edges[1:]) / 2
    tau_est = centers[np.argmax(counts)]
    tau_estimates.append(tau_est)

tau_estimates = np.array(tau_estimates)

# Todos deberían dar τ similar (misma propiedad del detector)
# Usamos la media como valor inicial y el mínimo tras el corte como cota superior
TAU_INIT = np.mean(tau_estimates)
TAU_MAX  = mins.min()    # mínimo real tras excluir pileup

print(f"τ por run (mod del histograma): {tau_estimates*1e6}")
print(f"TAU_INIT = {TAU_INIT*1e6:.1f} µs")
print(f"TAU_MAX  = {TAU_MAX*1e6:.1f} µs")

# Ya no usamos TAU_MAX como cota — los eventos en la zona prohibida
# (pileup) invalidan ese argumento

# Bounds para el optimizador basados en los estimados del modo
TAU_LO = 0.5 * TAU_INIT    # cota inferior conservadora
TAU_HI = 2.0 * TAU_INIT    # cota superior conservadora

print(f"Bounds para τ: [{TAU_LO*1e6:.1f}, {TAU_HI*1e6:.1f}] µs")

lam_init = 1.0 / (means - TAU_INIT)
print()
print(f"{'Run':>4}  {'λ_init (Hz)':>14}")
for i, lam in enumerate(lam_init):
    print(f'{i+1:>4}  {lam:>14.4f}')

τ por run (mod del histograma): [202.14166667 204.58       202.00833333 193.59166667 193.66666667
 192.54      ]
TAU_INIT = 198.1 µs
TAU_MAX  = 191.0 µs
Bounds para τ: [99.0, 396.2] µs

 Run     λ_init (Hz)
   1       3255.8305
   2       4881.1809
   3       7221.1257
   4      11575.9776
   5      20581.3316
   6      19252.3631


In [30]:
# Create new set of histograms for the clean data
# update with new deadtime estimation TAU_INIT
datasets_clean = [d[d>=TAU_INIT] for d in datasets_clean]
hists_original = hists
hists = []

for i, dt in enumerate(datasets_clean):
    h = ROOT.TH1D(f'h{i+1}_clean', f'Run {i+1};#Delta t (s);Counts / bin',
                  N_BINS, 0.0, T_MAX)
    # h.SetDirectory(0)      # keep alive outside any TFile
    h.SetLineColor(COLORS[i])
    h.SetLineWidth(1)
    for t in dt:
        if 0 < t <= T_MAX:
            h.Fill(t)
    hists.append(h)
    print(f'Run {i+1}: {h.GetEntries():.0f} entries filled')

Run 1: 449606 entries filled
Run 2: 437694 entries filled
Run 3: 404882 entries filled
Run 4: 334072 entries filled
Run 5: 173944 entries filled
Run 6: 140326 entries filled


Warning in <TROOT::Append>: Replacing existing TH1: h1_clean (Potential memory leak).


In [31]:
if ROOT.gROOT.FindObject('c_inspect'):
    ROOT.gROOT.FindObject('c_inspect').Close()

c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1600, 1200)
c_inspect.Divide(3, 2)
for i, h in enumerate(hists):
    c_inspect.cd(i + 1)
    h.SetLineWidth(1)
    h.SetLabelSize(0.04)
    h.Draw('HIST')
    c_inspect.cd(i+1).SetLogy()
c_inspect.Draw()

## Cell 7 — Individual ROOT fits (diagnostic)
Fit each histogram independently to $A\,e^{-\lambda(\Delta t - \tau)}$ to get per-run estimates of $\lambda_i$ and $\tau$.  
The spread in the fitted $\tau$ values tells you how consistent the detector model is.

> **Note on fit range:** we start the fit at `FIT_START = TAU_INIT * 1.05` to stay safely above the onset and avoid the partial first bin.

In [33]:
# Limpiar canvas y funciones anteriores para evitar conflictos de nombre
for name in ['c_indiv'] + [f'f_ind{i+1}' for i in range(N_RUNS)]:
    obj = ROOT.gROOT.FindObject(name)
    if obj:
        obj.Delete()


TAU_FROM_LED = 198e-6        # µs, del ajuste de eficiencia del LED
FIT_START    = TAU_FROM_LED * 1.10   # 211 µs — claramente sobre el onset
                                      # y sobre cualquier evento espurio

TAU_FIXED = 198e-6

ind_lambdas = []
ind_taus    = []
ind_errs_l  = []
ind_errs_t  = []
fit_funcs   = []

# T_FIT_MAXs = [3e-3, 2.3e-3, 1.7e-3, 1e-3, 0.4e-3, 0.3e-3]
T_FIT_MAXs = [3e-3, 2.3e-3, 1.7e-3, 1e-3, 1e-3, 1e-3]

c_indiv = ROOT.TCanvas('c_indiv', 'Individual fits', 1600, 1200)
c_indiv.Divide(3, 2)

for i, h in enumerate(hists):
    c_indiv.cd(i + 1)
    # ROOT.gPad.SetLogy(1)
    c_indiv.cd(i + 1).SetLogy()

    f = ROOT.TF1(f'f_ind{i+1}',
                 '[0] * exp( -[1] * (x - [2]) )',
                 FIT_START, T_FIT_MAXs[i])
    f.SetParNames('A', '#lambda', '#tau')

    # Initial values
    f.SetParameter(0, h.GetBinContent(h.FindBin(FIT_START * 1.1)))
    f.SetParameter(1, lam_init[i])
    f.SetParameter(2, TAU_FIXED)
    f.FixParameter(2, TAU_FIXED)

    # Bounds
    f.SetParLimits(0, 1e-3, 1e12)
    f.SetParLimits(1, 0.0,  1e12)
    f.SetParLimits(2, TAU_LO,  TAU_HI)

    h.Draw('HIST')
    # 'L' = log-likelihood fit (better for low-count bins), 'R' = use TF1 range
    h.Fit(f'f_ind{i+1}', 'LRQ')   # Q = quiet
    f.SetLineColor(ROOT.kBlack)
    f.SetLineWidth(2)
    f.Draw('SAME')
    fit_funcs.append(f)

    lam = f.GetParameter(1);  el = f.GetParError(1)
    tau = f.GetParameter(2);  et = f.GetParError(2)
    ind_lambdas.append(lam);  ind_errs_l.append(el)
    ind_taus.append(tau);     ind_errs_t.append(et)

c_indiv.Draw()

print(f"{'Run':>4}  {'λ (Hz)':>14}  {'δλ':>10}  {'τ (s)':>14}  {'δτ':>12}")
print('-' * 62)
for i in range(N_RUNS):
    print(f"{i+1:>4}  {ind_lambdas[i]:>14.4f}  {ind_errs_l[i]:>10.4f}"
          f"  {ind_taus[i]:>14.6e}  {ind_errs_t[i]:>12.4e}")

print(f"\nMean τ (individual fits) = {np.mean(ind_taus):.6e} s")
print(f"Std  τ (individual fits) = {np.std(ind_taus):.2e} s")

 Run          λ (Hz)          δλ           τ (s)            δτ
--------------------------------------------------------------
   1       3190.6076      4.9146    2.214690e-04    9.7953e-10
   2       4747.6849      7.4923    2.459120e-04    4.6610e-06
   3       7001.7618     11.7183    2.421650e-04    1.2611e-08
   4      11601.0941     22.4509    2.242306e-04    1.1389e-09
   5      17379.1650     54.8507    1.673145e-04    2.4628e-06
   6      15302.8134     55.3433    1.622619e-04    3.2070e-06

Mean τ (individual fits) = 2.105588e-04 s
Std  τ (individual fits) = 3.36e-05 s


Warning in <TCanvas::Delete()>: may not use this method
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_indiv


In [13]:
# Estimate from exponential shape only

tau_estimates = []
for i, dt in enumerate(datasets_clean):

    t_hi = T_FIT_MAXs[i]
    t_lo = min(TAU_INIT * 1.5, t_hi * 0.5)  # nunca supera la mitad del rango

    counts, edges = np.histogram(dt, bins=200, range=(T_SKIP, T_MAX))
    centers = (edges[:-1] + edges[1:]) / 2

    mask = (centers > t_lo) & (centers < t_hi) & (counts > 0)

    if mask.sum() < 3:
        print(f"Run {i+1}: rango limpio insuficiente "
              f"(t_lo={t_lo*1e6:.0f} µs, t_hi={t_hi*1e6:.0f} µs, "
              f"bins={mask.sum()})")
        tau_estimates.append(np.nan)
        continue

    coeffs    = np.polyfit(centers[mask], np.log(counts[mask]), 1)
    lambda_est = -coeffs[0]

    dt_range  = dt[(dt > T_SKIP) & (dt < t_hi)]
    mean_clean = dt_range.mean()

    tau_est   = mean_clean - 1.0 / lambda_est
    tau_estimates.append(tau_est)
    print(f"Run {i+1}: λ = {lambda_est:.1f} Hz,  τ = {tau_est*1e6:.1f} µs  "
          f"({mask.sum()} bins usados)")

tau_arr = np.array([t for t in tau_estimates if not np.isnan(t)])
print(f"\nMedia τ = {tau_arr.mean()*1e6:.1f} µs,  std = {tau_arr.std()*1e6:.1f} µs")

Run 1: λ = 3094.5 Hz,  τ = 188.9 µs  (149 bins usados)
Run 2: λ = 4657.6 Hz,  τ = 195.4 µs  (110 bins usados)
Run 3: λ = 6911.8 Hz,  τ = 197.1 µs  (77 bins usados)
Run 4: λ = 10573.2 Hz,  τ = 189.5 µs  (39 bins usados)
Run 5: λ = 18916.3 Hz,  τ = 182.1 µs  (11 bins usados)
Run 6: λ = 26392.1 Hz,  τ = 185.6 µs  (6 bins usados)

Media τ = 189.8 µs,  std = 5.2 µs
